## 1. Imports & Configuration

In [21]:
from __future__ import print_function
import random
import copy
import os
from ast import literal_eval
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from transformers import BertModel, BertForSequenceClassification, BertTokenizer

# Set random seeds for reproducibility
SEED = 1345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Model hyperparameters
fchidden = 256
hiddendim_lstm = 128
embeddim = 768
numlayers = 12

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


## 2. Data Loading Functions

Functions to load SemEval and Twitter datasets for ABSA.

In [ ]:
def load_semeval(dataset, num_classes, data_path=None):
    """ Loads SemEval 14 datasets

    Arguments:
        dataset (str): Name of the datsets
        num_classes (int): Number of classes to consider
        data_path (str): Path to the dataset

    """
    label = {'negative': 0,
             'positive': 1,
             'neutral': 2}

    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + 'atsa-' + dataset + '/atsa_train.json'
    test_file = data_path + 'atsa-' + dataset + '/atsa_test.json'

    temp = open(train_file, 'r', encoding='latin1').read()
    train = literal_eval(temp)
    train_sentence = []
    train_aspect = []
    train_sentiment = []
    for xml in train:
        train_sentence.append(xml['sentence'])
        train_aspect.append(xml['aspect'])
        train_sentiment.append(label[xml['sentiment']])

    temp = open(test_file, 'r', encoding='latin1').read()
    test = literal_eval(temp)
    test_sentence = []
    test_aspect = []
    test_sentiment = []
    for xml in test:
        test_sentence.append(xml['sentence'])
        test_aspect.append(xml['aspect'])
        test_sentiment.append(label[xml['sentiment']])

    train_sen_len = [len(sentence.split()) for sentence in train_sentence]
    train_asp_len = [len(aspect.split()) for aspect in train_aspect]

    print("----------------------------------------")
    print("Maximum Training data Sentence Length: {}".format(
                                                       max(train_sen_len)))
    print("Maximum Training data Aspect Length: {}".format(max(train_asp_len)))
    print("----------------------------------------")

    return (train_sentence, train_aspect, train_sentiment, test_sentence,
            test_aspect, test_sentiment)

In [23]:
def get_loader(input_ids, attention_masks, token_type_ids, labels, batchsize, shuffle=False):
    """Converts tensors into a dataloader."""
    array = utils.TensorDataset(input_ids, attention_masks, token_type_ids, labels)
    loader = utils.DataLoader(
        array,
        batch_size=batchsize,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )
    return loader

## 3. Tokenizer Functions

Functions to load pretrained BERT tokenizer and tokenize sentences.

In [ ]:
def get_pretrained_tokenizer():
    """
    Loads pretrained BERT Tokenizer and returns it,

    """
    print("Downloading bert tokenizer to cache")
    print("---------------------------------------")
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                                   do_lower_case=True
                                                   )
    return bert_tokenizer


def tokenize_sentences(bert_tokenizer, sentences, aspects, maxlen):
    """ converts sentences into ids according to bert tokenizer

    Arguments:
    bert_tokenizer (Tokenizer): Pretrained BERT Tokenizer
    sentences (list): List of sentences
    aspects (list): List of Aspects
    maxlen (int): Maximum Length of a sentence

    """
    input_ids = []
    
    attention_masks = []
    token_type_ids = []

    for sentence, aspect in zip(sentences, aspects):

        encoded = bert_tokenizer(
            text=sentence,
            text_pair=aspect,
            add_special_tokens=True,
            max_length=maxlen,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors='pt'
        )

        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
        token_type_ids.append(encoded['token_type_ids'])

    input_ids = torch.cat(input_ids, dim=0, out=None)
    attention_masks = torch.cat(attention_masks, dim=0, out=None)
    token_type_ids = torch.cat(token_type_ids, dim=0, out=None)

    return input_ids, attention_masks, token_type_ids

## 4. Model Definitions

BERT-based models for ABSA:
- **Bert_Base**: Using BertForSequenceClassification
- **Bert_LSTM**: BERT + LSTM
- **Bert_Attention**: BERT + attention over hidden layers
- **Bert_LSTM_Attention**: BERT token outputs + BiLSTM + token attention

In [25]:
class Bert_Base(nn.Module):
    def __init__(self, numclasses):
        super(Bert_Base, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.dropout = nn.Dropout(0.1)

        self.bert = BertForSequenceClassification.from_pretrained('bert-base-uncased',
                                                                   output_hidden_states=False,
                                                                   output_attentions=False,
                                                                   num_labels=self.numclasses)
        print("BERT Model Loaded")

    def forward(self, inp_ids, att_mask, token_ids, labels):
        outputs = self.bert(input_ids=inp_ids, attention_mask=att_mask,
                            token_type_ids=token_ids, labels=labels)
        return outputs.loss, outputs.logits

In [26]:
class Bert_LSTM(nn.Module):
    def __init__(self, numclasses):
        super(Bert_LSTM, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.hiddendim_lstm = hiddendim_lstm
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")
        self.lstm = nn.LSTM(self.embeddim, self.hiddendim_lstm, batch_first=True)
        self.fc = nn.Linear(self.hiddendim_lstm, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out, _ = self.lstm(hidden_states, None)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [27]:
class Bert_Attention(nn.Module):
    def __init__(self, numclasses, device=None):
        super(Bert_Attention, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.fchidden = fchidden
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")

        q_t = np.random.normal(loc=0.0, scale=0.1, size=(1, self.embeddim))
        self.q = nn.Parameter(torch.from_numpy(q_t).float())

        w_ht = np.random.normal(loc=0.0, scale=0.1, size=(self.embeddim, self.fchidden))
        self.w_h = nn.Parameter(torch.from_numpy(w_ht).float())

        self.fc = nn.Linear(self.fchidden, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out = self.attention(hidden_states)
        out = self.dropout(out)
        out = self.fc(out)
        return out

    def attention(self, h):
        v = torch.matmul(self.q, h.transpose(-2, -1)).squeeze(1)
        v = F.softmax(v, -1)
        v_temp = torch.matmul(v.unsqueeze(1), h).transpose(-2, -1)
        v = torch.matmul(self.w_h.transpose(1, 0), v_temp).squeeze(2)
        return v


class Bert_LSTM_Attention(nn.Module):
    def __init__(self, numclasses, lstm_hidden=128, dropout=0.1):
        super(Bert_LSTM_Attention, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.lstm_hidden = lstm_hidden

        self.bert = BertModel.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=False,
            output_attentions=False
        )
        print('BERT Model Loaded (LSTM+Attention)')

        self.lstm = nn.LSTM(
            input_size=self.embeddim,
            hidden_size=self.lstm_hidden,
            batch_first=True,
            bidirectional=True
        )
        self.attn = nn.Linear(self.lstm_hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.lstm_hidden * 2, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(
            input_ids=inp_ids,
            attention_mask=att_mask,
            token_type_ids=token_ids
        )

        sequence_output = outputs.last_hidden_state
        lstm_out, _ = self.lstm(sequence_output)

        attn_scores = self.attn(lstm_out).squeeze(-1)
        mask_value = torch.finfo(attn_scores.dtype).min
        attn_scores = attn_scores.masked_fill(att_mask == 0, mask_value)
        attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(1)

        context = torch.bmm(attn_weights, lstm_out).squeeze(1)
        logits = self.fc(self.dropout(context))
        return logits

## 5. Training & Evaluation Functions

Functions for training the model and evaluating performance.

In [28]:
def evaluate(loader, net, device, model_name):
    """ Evaluates a model and returns loss, accuracy

    Arguments:
    loader (DataLoader): dataloader to evaluate
    net (nn.Module): Model to evaluate
    device (torch.device): Device type
    model_name (str): Model name

    """
    net.eval()

    with torch.no_grad():
        loss = 0.0
        total = 0.0
        acc = 0.0
        y_pred = []
        y_true = []
        for input_id, attention_masks, token_ids, labels in loader:
            input_id = input_id.to(device)
            attention_masks = attention_masks.to(device)
            token_ids = token_ids.to(device)
            labels = labels.long().to(device)

            if(model_name == 'base'):
                curloss, output = net(input_id, attention_masks, token_ids, labels)
            else:
                output = net(input_id, attention_masks, token_ids)
                curloss = F.cross_entropy(output, labels, reduction='sum')
            loss += curloss.item()
            preds = torch.argmax(output, 1)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.tolist())
            acc += torch.sum(preds == labels).item()
            total += input_id.size(0)

        F1 = round((f1_score(y_true, y_pred, average='macro')), 2) * 100
        return round((loss / total), 3), round(((acc / total) * 100), 2), F1

In [ ]:
def build_model(model_name, numclasses, device):
    if model_name == 'lstm':
        return Bert_LSTM(numclasses).to(device)
    if model_name == 'attention':
        return Bert_Attention(numclasses, device).to(device)
    if model_name == 'lstm_attention':
        return Bert_LSTM_Attention(numclasses).to(device)
    return Bert_Base(numclasses).to(device)


# def maybe_freeze_bert_layers(model, freeze_bert=False, freeze_n_layers=0):
#     if not freeze_bert:
#         return

#     # Handle both BertModel-based and BertForSequenceClassification-based wrappers.
#     bert_backbone = None
#     if hasattr(model, 'bert') and hasattr(model.bert, 'encoder'):
#         bert_backbone = model.bert
#     elif hasattr(model, 'bert') and hasattr(model.bert, 'bert'):
#         bert_backbone = model.bert.bert

#     if bert_backbone is None:
#         return

#     for p in bert_backbone.embeddings.parameters():
#         p.requires_grad = False

#     for layer_idx, layer in enumerate(bert_backbone.encoder.layer):
#         if layer_idx < freeze_n_layers:
#             for p in layer.parameters():
#                 p.requires_grad = False


def train_model_kfold(input_ids, attention_masks, token_type_ids, labels, test_loader,
                      model_name, numclasses, numepochs, k_folds, batch_size, device,
                      save_prefix='kfold',
                      early_stopping_patience=2,
                      use_amp=True,
                      freeze_bert=False,
                      freeze_n_layers=0,
                      skip_existing_folds=True,
                      max_folds_to_run=None):
    """Train a model using stratified K-fold CV and save each fold checkpoint."""
    labels_np = labels.cpu().numpy()
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)

    fold_results = []
    saved_paths = []
    best_overall_acc = -1.0
    best_overall_state_dict = None

    amp_enabled = use_amp and (device.type == 'cuda')

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_np)), labels_np), start=1):
        if max_folds_to_run is not None and fold_idx > max_folds_to_run:
            print(f"Stopping early due to max_folds_to_run={max_folds_to_run}")
            break

        print(f"\n{'=' * 72}")
        print(f"TRAINING {model_name.upper()} - FOLD {fold_idx}/{k_folds}")
        print(f"{'=' * 72}")

        fold_save_path = f"{save_prefix}_{model_name}_fold{fold_idx}.pth"

        # Resume mode: skip fold if checkpoint exists.
        if skip_existing_folds and os.path.exists(fold_save_path):
            print(f"Skipping fold {fold_idx}, found existing checkpoint: {fold_save_path}")
            checkpoint = torch.load(fold_save_path, map_location=device)
            model = build_model(model_name, numclasses, device)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.eval()

            _, test_acc, test_f1 = evaluate(test_loader, model, device, model_name)
            fold_results.append({
                'fold': fold_idx,
                'val_acc': None,
                'test_acc': test_acc,
                'test_f1': test_f1
            })
            saved_paths.append(fold_save_path)

            if test_acc > best_overall_acc:
                best_overall_acc = test_acc
                best_overall_state_dict = copy.deepcopy(model.state_dict())
            continue

        train_idx_t = torch.tensor(train_idx, dtype=torch.long)
        val_idx_t = torch.tensor(val_idx, dtype=torch.long)

        fold_train_loader = get_loader(
            input_ids[train_idx_t],
            attention_masks[train_idx_t],
            token_type_ids[train_idx_t],
            labels[train_idx_t],
            batch_size,
            shuffle=True
        )
        fold_val_loader = get_loader(
            input_ids[val_idx_t],
            attention_masks[val_idx_t],
            token_type_ids[val_idx_t],
            labels[val_idx_t],
            batch_size,
            shuffle=False
        )

        model = build_model(model_name, numclasses, device)
        maybe_freeze_bert_layers(model, freeze_bert=freeze_bert, freeze_n_layers=freeze_n_layers)

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=2e-5
        )
        scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)

        best_val_acc = -1.0
        best_fold_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        for epoch in range(1, numepochs + 1):
            model.train()
            for input_id, att_masks, token_ids, batch_labels in fold_train_loader:
                input_id = input_id.to(device)
                att_masks = att_masks.to(device)
                token_ids = token_ids.to(device)
                batch_labels = batch_labels.long().to(device)

                model.zero_grad()

                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    if model_name == 'base':
                        loss, _ = model(input_id, att_masks, token_ids, batch_labels)
                    else:
                        output = model(input_id, att_masks, token_ids)
                        loss = F.cross_entropy(output, batch_labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            val_loss, val_acc, val_f1 = evaluate(fold_val_loader, model, device, model_name)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_fold_state_dict = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            print(f"Fold {fold_idx} Epoch {epoch} Val Loss {val_loss} Val Acc {val_acc} Val F1 {val_f1}")

            if epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch} (patience={early_stopping_patience})")
                break

        model.load_state_dict(best_fold_state_dict)
        _, test_acc, test_f1 = evaluate(test_loader, model, device, model_name)

        fold_results.append({
            'fold': fold_idx,
            'val_acc': best_val_acc,
            'test_acc': test_acc,
            'test_f1': test_f1
        })

        torch.save({
            'model_state_dict': best_fold_state_dict,
            'model_type': model_name,
            'numclasses': numclasses,
            'maxlen': maxlen,
            'fold': fold_idx,
            'k_folds': k_folds
        }, fold_save_path)
        saved_paths.append(fold_save_path)

        print(f"Saved fold {fold_idx} model to: {fold_save_path}")
        print(f"Fold {fold_idx} Test Accuracy {test_acc} F1 {test_f1}")

        if test_acc > best_overall_acc:
            best_overall_acc = test_acc
            best_overall_state_dict = copy.deepcopy(best_fold_state_dict)

    valid_folds = [x for x in fold_results if x.get('test_acc') is not None]
    avg_test_acc = np.mean([x['test_acc'] for x in valid_folds]) if len(valid_folds) > 0 else 0.0
    avg_test_f1 = np.mean([x['test_f1'] for x in valid_folds]) if len(valid_folds) > 0 else 0.0
    print(f"\nAverage {model_name.upper()} Test Accuracy across completed folds: {avg_test_acc:.2f}")
    print(f"Average {model_name.upper()} Test F1 across completed folds: {avg_test_f1:.2f}")

    best_model = build_model(model_name, numclasses, device)
    if best_overall_state_dict is not None:
        best_model.load_state_dict(best_overall_state_dict)
    best_model.eval()

    best_model_save_path = f"{save_prefix}_best_{model_name}_model.pth"
    if best_overall_state_dict is not None:
        torch.save({
            'model_state_dict': best_overall_state_dict,
            'model_type': model_name,
            'numclasses': numclasses,
            'maxlen': maxlen,
            'k_folds': k_folds
        }, best_model_save_path)
        saved_paths.append(best_model_save_path)
        print(f"Saved best {model_name.upper()} model to: {best_model_save_path}")

    return best_model, fold_results, saved_paths

## 6. Main Pipeline

Configure the parameters and run the training pipeline.

In [ ]:
dataset = 'laptop'          
maxlen = 80                
numclasses = 3              
model_name = 'lstm'         # Options: 'lstm', 'attention', 'base', 'lstm_attention'
data_path = '../Data/'      # Path to dataset folder
batch_size = 8              # Batch size
numepochs = 4               # Reduced epochs for faster review runs
k_folds = 5                 # Keep 5-fold CV

early_stopping_patience = 2
use_amp = True              # Mixed precision on GPU
freeze_bert = True          # Freeze lower BERT layers for faster training
freeze_n_layers = 8         # Number of encoder layers to freeze
skip_existing_folds = True  # Resume capability: skip already saved fold checkpoints
max_folds_to_run = None     # Set to 2/3 for ultra-fast partial run

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


In [31]:
# Load dataset
if dataset in ['laptop', 'restaurant']:
    print('semeval')
    (train_sentence, train_aspect, train_sentiment,
     test_sentence, test_aspect, test_sentiment) = load_semeval(dataset,
                                                                numclasses,
                                                                data_path)
'''
elif dataset == 'twitter':
    print('twitter')
    (train_sentence, train_aspect, train_sentiment,
     test_sentence, test_aspect, test_sentiment) = load_twitter(dataset,
                                                                data_path)'''

semeval
----------------------------------------
Maximum Training data Sentence Length: 78
Maximum Training data Aspect Length: 6
----------------------------------------


"\nelif dataset == 'twitter':\n    print('twitter')\n    (train_sentence, train_aspect, train_sentiment,\n     test_sentence, test_aspect, test_sentiment) = load_twitter(dataset,\n                                                                data_path)"

In [32]:
# Split training data into train and validation sets
(train_sentence, dev_sentence, train_aspect, dev_aspect,
 train_sentiment, dev_sentiment) = train_test_split(train_sentence,
                                                    train_aspect,
                                                    train_sentiment,
                                                    test_size=0.1,
                                                    random_state=42)

print("Training Data size: {}".format(len(train_sentence)))
print("Validation Data size: {}".format(len(dev_sentence)))
print("Test Data size: {}".format(len(test_sentence)))
print("------------------------------------------------")

Training Data size: 2081
Validation Data size: 232
Test Data size: 638
------------------------------------------------


In [33]:
# Load pretrained BERT tokenizer
bert_tokenizer = get_pretrained_tokenizer()

---------------------------------------


In [34]:
# Tokenize training data
print("Tokenizing training data")
(train_input_ids, train_attention_masks,
 train_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                            train_sentence,
                                            train_aspect,
                                            maxlen)
train_labels = torch.from_numpy(np.asarray(train_sentiment, 'int32'))

# Tokenize validation data
print("Tokenizing validation data")
(dev_input_ids, dev_attention_masks,
 dev_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                          dev_sentence,
                                          dev_aspect,
                                          maxlen)
dev_labels = torch.from_numpy(np.asarray(dev_sentiment, 'int32'))

# Tokenize test data
print("Tokenizing test data")
(test_input_ids, test_attention_masks,
 test_token_type_ids) = tokenize_sentences(bert_tokenizer,
                                           test_sentence,
                                           test_aspect,
                                           maxlen)
test_labels = torch.from_numpy(np.asarray(test_sentiment, 'int32'))
print("-------------------------------------------------")

Tokenizing training data
Tokenizing validation data
Tokenizing test data
-------------------------------------------------


In [35]:
# Create data loaders
train_loader = get_loader(train_input_ids, train_attention_masks,
                          train_token_type_ids, train_labels,
                          batch_size, shuffle=True)
dev_loader = get_loader(dev_input_ids, dev_attention_masks,
                        dev_token_type_ids, dev_labels,
                        batch_size, shuffle=False)
test_loader = get_loader(test_input_ids, test_attention_masks,
                         test_token_type_ids, test_labels,
                         batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(dev_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 261
Validation batches: 29
Test batches: 80


In [36]:
# Train all models with 5-fold cross-validation and store the best checkpoints
model_names = ['lstm', 'attention', 'base', 'lstm_attention']
trained_models = {}
model_results = {}
saved_model_paths = {}
all_fold_paths = {}

for model_name in model_names:
    print("\n" + "=" * 70)
    print(f"TRAINING {model_name.upper()} MODEL WITH {k_folds}-FOLD CV")
    print("=" * 70)

    trained_model, fold_results, fold_paths = train_model_kfold(
        input_ids=train_input_ids,
        attention_masks=train_attention_masks,
        token_type_ids=train_token_type_ids,
        labels=train_labels,
        test_loader=test_loader,
        model_name=model_name,
        numclasses=numclasses,
        numepochs=numepochs,
        k_folds=k_folds,
        batch_size=batch_size,
        device=DEVICE,
        save_prefix='kfold',
        early_stopping_patience=early_stopping_patience,
        use_amp=use_amp,
        freeze_bert=freeze_bert,
        freeze_n_layers=freeze_n_layers,
        skip_existing_folds=skip_existing_folds,
        max_folds_to_run=max_folds_to_run
    )

    trained_models[model_name] = trained_model
    all_fold_paths[model_name] = [p for p in fold_paths if f'_{model_name}_fold' in p]
    best_path = f'kfold_best_{model_name}_model.pth'
    saved_model_paths[model_name] = best_path

    # Evaluate best checkpoint on full train/dev/test sets for comparison table
    train_loss, train_acc, train_f1 = evaluate(train_loader, trained_model, DEVICE, model_name)
    val_loss, val_acc, val_f1 = evaluate(dev_loader, trained_model, DEVICE, model_name)
    test_loss, test_acc, test_f1 = evaluate(test_loader, trained_model, DEVICE, model_name)

    valid_fold_rows = [x for x in fold_results if x.get('test_acc') is not None]
    kfold_avg_test_acc = round(float(np.mean([x['test_acc'] for x in valid_fold_rows])), 2) if len(valid_fold_rows) > 0 else None
    kfold_avg_test_f1 = round(float(np.mean([x['test_f1'] for x in valid_fold_rows])), 2) if len(valid_fold_rows) > 0 else None

    model_results[model_name] = {
        'train_acc': train_acc,
        'train_f1': train_f1,
        'val_acc': val_acc,
        'val_f1': val_f1,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'kfold_avg_test_acc': kfold_avg_test_acc,
        'kfold_avg_test_f1': kfold_avg_test_f1
    }

# Print comparison of all models
print("\n" + "=" * 80)
print(f"MODEL COMPARISON SUMMARY ({k_folds}-FOLD CV)")
print("=" * 80)
print(f"\n{'Model':<16} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12} {'KF Avg Acc':<12} {'KF Avg F1':<12}")
print("-" * 90)
for name, results in model_results.items():
    print(
        f"{name.upper():<16} {results['train_acc']:<12} {results['val_acc']:<12} "
        f"{results['test_acc']:<12} {str(results['kfold_avg_test_acc']):<12} {str(results['kfold_avg_test_f1']):<12}"
    )

best_model_name = max(
    model_results,
    key=lambda x: model_results[x]['kfold_avg_test_acc'] if model_results[x]['kfold_avg_test_acc'] is not None else -1
)
print(f"\nBest model by KFold Avg Test Acc: {best_model_name.upper()} ({model_results[best_model_name]['kfold_avg_test_acc']}%)")

print("\n" + "=" * 80)
print("SAVED KFOLD MODEL FILES")
print("=" * 80)
for name, path in saved_model_paths.items():
    print(f"  - BEST {name.upper():<14}: {path}")
for name, paths in all_fold_paths.items():
    for path in paths:
        print(f"  - FOLD {name.upper():<14}: {path}")


TRAINING LSTM MODEL WITH 5-FOLD CV

TRAINING LSTM - FOLD 1/5
Skipping fold 1, found existing checkpoint: kfold_lstm_fold1.pth


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4367.49it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded

TRAINING LSTM - FOLD 2/5
Skipping fold 2, found existing checkpoint: kfold_lstm_fold2.pth


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8465.44it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded

TRAINING LSTM - FOLD 3/5
Skipping fold 3, found existing checkpoint: kfold_lstm_fold3.pth


...

BERT Model Loaded (LSTM+Attention)

TRAINING LSTM_ATTENTION - FOLD 5/5
Skipping fold 5, found existing checkpoint: kfold_lstm_attention_fold5.pth


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9903.96it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded (LSTM+Attention)

Average LSTM_ATTENTION Test Accuracy across completed folds: 76.30
Average LSTM_ATTENTION Test F1 across completed folds: 70.40


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12438.77it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded (LSTM+Attention)
Saved best LSTM_ATTENTION model to: kfold_best_lstm_attention_model.pth

MODEL COMPARISON SUMMARY (5-FOLD CV)

Model            Train Acc    Val Acc      Test Acc     KF Avg Acc   KF Avg F1   
------------------------------------------------------------------------------------------
LSTM             88.27        74.14        77.74        75.8         69.0        
ATTENTION        87.79        75.86        77.43        76.65        70.8        
BASE             89.76        71.98        76.96        76.24        71.2        
LSTM_ATTENTION   87.41        75.43        78.21        76.3         70.4        

Best model by KFold Avg Test Acc: ATTENTION (76.65%)

SAVED KFOLD MODEL FILES
  - BEST LSTM          : kfold_best_lstm_model.pth
  - BEST ATTENTION     : kfold_best_attention_model.pth
  - BEST BASE          : kfold_best_base_model.pth
  - BEST LSTM_ATTENTION: kfold_best_lstm_attention_model.pth
  - FOLD LSTM          : kfold_lstm_fold1.pth
  - FOLD 

## 7. Save Models

Save all 3 trained models to separate files.

In [37]:
# Save all best kfold models to dedicated kfold files (already saved during training).
# This cell prints the saved paths again for convenience.
print("\nBest KFold model checkpoints:")
for name, path in saved_model_paths.items():
    print(f"  - {name.upper()}: {path}")


Best KFold model checkpoints:
  - LSTM: kfold_best_lstm_model.pth
  - ATTENTION: kfold_best_attention_model.pth
  - BASE: kfold_best_base_model.pth
  - LSTM_ATTENTION: kfold_best_lstm_attention_model.pth


## 8. Inference

Load models from saved files and perform inference.

In [38]:
# Function to load a saved model
def load_model(model_path, device):
    """Load a saved model for inference."""
    checkpoint = torch.load(model_path, map_location=device)

    model_type = checkpoint['model_type']
    num_classes = checkpoint['numclasses']
    max_len = checkpoint['maxlen']

    if model_type == 'lstm':
        model = Bert_LSTM(num_classes).to(device)
    elif model_type == 'attention':
        model = Bert_Attention(num_classes, device).to(device)
    elif model_type == 'lstm_attention':
        model = Bert_LSTM_Attention(num_classes).to(device)
    else:
        model = Bert_Base(num_classes).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    return model, max_len, model_type


# Prediction function
def predict_sentiment(model, bert_tokenizer, sentence, aspect, maxlen, device, model_type='lstm'):
    """Predict sentiment for a sentence-aspect pair."""
    model.eval()

    encoded = bert_tokenizer(
        text=sentence,
        text_pair=aspect,
        add_special_tokens=True,
        max_length=maxlen,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt'
    )

    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    token_type_ids = encoded['token_type_ids'].to(device)

    with torch.no_grad():
        if model_type == 'base':
            dummy_label = torch.tensor([0]).to(device)
            _, output = model(input_ids, attention_mask, token_type_ids, dummy_label)
        else:
            output = model(input_ids, attention_mask, token_type_ids)

        probabilities = F.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    return predicted.item(), confidence.item()


# Label mapping
label_names = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}

### 8.1 LSTM Model Inference

In [39]:
# Load LSTM model from saved KFold file
lstm_model_path = 'kfold_best_lstm_model.pth'
lstm_model, lstm_maxlen, lstm_type = load_model(lstm_model_path, DEVICE)
print(f"Loaded LSTM model from: {lstm_model_path}")

# Test examples
test_examples = [
    ("The battery life of this laptop is amazing and lasts all day.", "battery life"),
    ("The screen resolution is terrible and hurts my eyes.", "screen resolution"),
    ("The keyboard is okay, nothing special.", "keyboard"),
    ("I love the fast performance of this machine.", "performance"),
    ("The customer service was unhelpful and rude.", "customer service"),
]

print("\n" + "=" * 70)
print("LSTM MODEL PREDICTIONS (KFOLD BEST)")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(lstm_model, bert_tokenizer,
                                                sentence, aspect, lstm_maxlen, DEVICE,
                                                model_type=lstm_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10473.92it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded
Loaded LSTM model from: kfold_best_lstm_model.pth

LSTM MODEL PREDICTIONS (KFOLD BEST)

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 97.04%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 93.76%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 84.29%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 97.27%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 91.17%)


### 8.2 Attention Model Inference

In [40]:
# Load Attention model from saved KFold file
attention_model_path = 'kfold_best_attention_model.pth'
attention_model, attention_maxlen, attention_type = load_model(attention_model_path, DEVICE)
print(f"Loaded ATTENTION model from: {attention_model_path}")

print("\n" + "=" * 70)
print("ATTENTION MODEL PREDICTIONS (KFOLD BEST)")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(attention_model, bert_tokenizer,
                                                sentence, aspect, attention_maxlen, DEVICE,
                                                model_type=attention_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9646.09it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded
Loaded ATTENTION model from: kfold_best_attention_model.pth

ATTENTION MODEL PREDICTIONS (KFOLD BEST)

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 99.50%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 98.97%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 78.19%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 99.73%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 96.16%)


### 8.3 Base Model Inference

In [ ]:
base_model_path = 'kfold_best_base_model.pth'
base_model, base_maxlen, base_type = load_model(base_model_path, DEVICE)
print(f"Loaded BASE model from: {base_model_path}")

print("\n" + "=" * 70)
print("BASE MODEL PREDICTIONS (KFOLD BEST)")
print("=" * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(base_model, bert_tokenizer,
                                                sentence, aspect, base_maxlen, DEVICE,
                                                model_type=base_type)
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9810.72it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

BERT Model Loaded
Loaded BASE model from: kfold_best_base_model.pth

BASE MODEL PREDICTIONS (KFOLD BEST)

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 99.04%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 94.76%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 60.74%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 99.14%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 98.74%)


### 8.4 Compare All Models

In [42]:
# Compare predictions from all 4 models side-by-side
print("=" * 90)
print("COMPARISON: KFOLD BEST MODELS ON SAME EXAMPLES")
print("=" * 90)

# Load kfold best LSTM+Attention model for comparison
lstm_attention_path = 'kfold_best_lstm_attention_model.pth'
lstm_attention_loaded, lstm_attention_maxlen, lstm_attention_type = load_model(lstm_attention_path, DEVICE)

loaded_models = {
    'lstm': (lstm_model, lstm_maxlen, lstm_type),
    'attention': (attention_model, attention_maxlen, attention_type),
    'base': (base_model, base_maxlen, base_type),
    'lstm_attention': (lstm_attention_loaded, lstm_attention_maxlen, lstm_attention_type)
}

for sentence, aspect in test_examples:
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print("-" * 80)

    for model_name, (model, max_len, model_type) in loaded_models.items():
        pred_label, confidence = predict_sentiment(model, bert_tokenizer,
                                                    sentence, aspect, max_len, DEVICE,
                                                    model_type=model_type)
        print(f"  {model_name.upper():<16}: {label_names[pred_label]:<10} (confidence: {confidence:.2%})")

COMPARISON: KFOLD BEST MODELS ON SAME EXAMPLES


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9791.15it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT Model Loaded (LSTM+Attention)

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
--------------------------------------------------------------------------------
  LSTM            : Positive   (confidence: 97.04%)
  ATTENTION       : Positive   (confidence: 99.50%)
  BASE            : Positive   (confidence: 99.04%)
  LSTM_ATTENTION  : Positive   (confidence: 97.65%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
--------------------------------------------------------------------------------
  LSTM            : Negative   (confidence: 93.76%)
  ATTENTION       : Negative   (confidence: 98.97%)
  BASE            : Negative   (confidence: 94.76%)
  LSTM_ATTENTION  : Negative   (confidence: 97.04%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
--------------------------------------------------------------------------------
  LSTM            : Positive   (confidence:

## 9. Additional Model: BERT + BiLSTM + Attention

This section adds a combined architecture where token representations from BERT are encoded with BiLSTM, then pooled using attention over tokens.

In [43]:
# Bert_LSTM_Attention is already defined in Section 4.
# This section continues with loading/evaluation for the KFold checkpoint.

In [44]:
# Load the KFold-trained BERT+BiLSTM+Attention model
lstm_attention_model_path = 'kfold_best_lstm_attention_model.pth'

if 'lstm_attention' in trained_models:
    lstm_attention_model = trained_models['lstm_attention']
    print(f'Using in-memory KFold trained model: {lstm_attention_model_path}')
elif os.path.exists(lstm_attention_model_path):
    checkpoint = torch.load(lstm_attention_model_path, map_location=DEVICE)
    lstm_attention_model = Bert_LSTM_Attention(numclasses).to(DEVICE)
    lstm_attention_model.load_state_dict(checkpoint['model_state_dict'])
    lstm_attention_model.eval()
    print(f'Loaded KFold model from: {lstm_attention_model_path}')
else:
    raise FileNotFoundError(
        'kfold_best_lstm_attention_model.pth not found. Run the KFold training cell first.'
    )

train_loss, train_acc, train_f1 = evaluate(train_loader, lstm_attention_model, DEVICE, model_name='lstm_attention')
val_loss, val_acc, val_f1 = evaluate(dev_loader, lstm_attention_model, DEVICE, model_name='lstm_attention')
test_loss, test_acc, test_f1 = evaluate(test_loader, lstm_attention_model, DEVICE, model_name='lstm_attention')

print('\n' + '=' * 70)
print('BERT + BiLSTM + Attention Results (KFold Best)')
print('=' * 70)
print(f'Train Acc: {train_acc} | Train F1: {train_f1}')
print(f'Val   Acc: {val_acc} | Val   F1: {val_f1}')
print(f'Test  Acc: {test_acc} | Test  F1: {test_f1}')

Using in-memory KFold trained model: kfold_best_lstm_attention_model.pth

BERT + BiLSTM + Attention Results (KFold Best)
Train Acc: 87.41 | Train F1: 85.0
Val   Acc: 75.43 | Val   F1: 74.0
Test  Acc: 78.21 | Test  F1: 73.0


In [45]:
# Quick inference with the new BERT+BiLSTM+Attention model
print('\n' + '=' * 70)
print('LSTM+ATTENTION MODEL PREDICTIONS')
print('=' * 70)

for sentence, aspect in test_examples:
    pred_label, confidence = predict_sentiment(
        lstm_attention_model,
        bert_tokenizer,
        sentence,
        aspect,
        maxlen,
        DEVICE,
        model_type='lstm_attention'
    )
    print(f'\nSentence: "{sentence}"')
    print(f'Aspect: "{aspect}"')
    print(f'Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})')


LSTM+ATTENTION MODEL PREDICTIONS

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
Prediction: Positive (confidence: 97.65%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
Prediction: Negative (confidence: 97.04%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
Prediction: Positive (confidence: 60.14%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
Prediction: Positive (confidence: 98.19%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
Prediction: Negative (confidence: 94.22%)


## 10. Parameter-Efficient Fine-Tuning: BERT Base with QLoRA

This section fine-tunes `bert-base-uncased` using 4-bit quantization + LoRA adapters (QLoRA) for sequence classification.

In [65]:
# If needed, install once in your environment:
# !pip install -U transformers peft bitsandbytes accelerate datasets tf-keras

# Prefer PyTorch-only code path for this notebook's QLoRA section.
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['USE_TF'] = '0'

qlora_ready = True
try:
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        Trainer,
        TrainingArguments,
        DataCollatorWithPadding,
        BitsAndBytesConfig
    )
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
except Exception as e:
    qlora_ready = False
    print('QLoRA import/setup issue:', repr(e))
    print('Try: pip install -U transformers peft bitsandbytes accelerate datasets tf-keras')
    print('If you changed env vars, restart the kernel and run this cell again.')

print(f'QLoRA ready: {qlora_ready}')

QLoRA ready: True


In [66]:
def build_hf_absa_dataset(sentences, aspects, labels):
    return Dataset.from_dict({
        'text': sentences,
        'text_pair': aspects,
        'labels': labels
    })


def compute_metrics_macro_f1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    acc = (preds == labels).mean()
    return {
        'accuracy': float(acc),
        'macro_f1': float(macro_f1)
    }

In [67]:
def run_bert_base_qlora(train_sentence, train_aspect, train_sentiment,
                        dev_sentence, dev_aspect, dev_sentiment,
                        test_sentence, test_aspect, test_sentiment,
                        numclasses=3,
                        model_name='bert-base-uncased',
                        max_length=128,
                        output_dir='kfold_bert_base_qlora_single',
                        # Tunable training knobs for better QLoRA quality
                        num_train_epochs=6,
                        learning_rate=5e-5,
                        per_device_train_batch_size=8,
                        per_device_eval_batch_size=16,
                        gradient_accumulation_steps=2,
                        warmup_ratio=0.06,
                        weight_decay=0.01,
                        qlora_4bit=True,
                        lora_r=16,
                        lora_alpha=32,
                        lora_dropout=0.05):
    if not qlora_ready:
        raise RuntimeError('QLoRA dependencies are not installed.')

    if not torch.cuda.is_available():
        raise RuntimeError('QLoRA with bitsandbytes 4-bit requires CUDA GPU support.')

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    train_ds = build_hf_absa_dataset(train_sentence, train_aspect, train_sentiment)
    dev_ds = build_hf_absa_dataset(dev_sentence, dev_aspect, dev_sentiment)
    test_ds = build_hf_absa_dataset(test_sentence, test_aspect, test_sentiment)

    def tokenize_batch(batch):
        return tokenizer(
            batch['text'],
            batch['text_pair'],
            truncation=True,
            max_length=max_length
        )

    train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=['text', 'text_pair'])
    dev_ds = dev_ds.map(tokenize_batch, batched=True, remove_columns=['text', 'text_pair'])
    test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=['text', 'text_pair'])

    def build_peft_model(use_4bit=True):
        if use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16
            )
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=numclasses,
                quantization_config=bnb_config,
                device_map='auto'
            )
            model = prepare_model_for_kbit_training(model)
        else:
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=numclasses
            ).to('cuda')

        model.config.use_cache = False
        model.gradient_checkpointing_enable()

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias='none',
            # Include key for better adapter capacity on ABSA.
            target_modules=['query', 'key', 'value'],
            task_type=TaskType.SEQ_CLS
        )

        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

        if use_4bit:
            model = model.to('cuda')

        return model

    def build_training_args(prefer_paged_optim=True):
        base_training_kwargs = {
            'output_dir': output_dir,
            'num_train_epochs': num_train_epochs,
            'per_device_train_batch_size': per_device_train_batch_size,
            'per_device_eval_batch_size': per_device_eval_batch_size,
            'gradient_accumulation_steps': gradient_accumulation_steps,
            'learning_rate': learning_rate,
            'warmup_ratio': warmup_ratio,
            'weight_decay': weight_decay,
            'logging_steps': 20,
            'save_strategy': 'epoch',
            'save_total_limit': 1,
            'load_best_model_at_end': True,
            'metric_for_best_model': 'macro_f1',
            'greater_is_better': True,
            'fp16': True,
            'report_to': 'none',
            'disable_tqdm': True,
            'lr_scheduler_type': 'cosine',
            'max_grad_norm': 1.0
        }

        optim_order = ['paged_adamw_8bit', 'adamw_torch'] if prefer_paged_optim else ['adamw_torch']

        training_args = None
        last_error = None
        for optim_name in optim_order:
            for eval_key in ['evaluation_strategy', 'eval_strategy']:
                candidate_kwargs = dict(base_training_kwargs)
                candidate_kwargs['optim'] = optim_name
                candidate_kwargs[eval_key] = 'epoch'
                try:
                    training_args = TrainingArguments(**candidate_kwargs)
                    print(f'Using TrainingArguments with {eval_key} and optim={optim_name}')
                    break
                except TypeError as e:
                    last_error = e
            if training_args is not None:
                break

        if training_args is None:
            raise last_error

        return training_args

    def build_trainer(model, training_args):
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

        trainer = None
        trainer_error = None
        trainer_base_kwargs = {
            'model': model,
            'args': training_args,
            'train_dataset': train_ds,
            'eval_dataset': dev_ds,
            'data_collator': data_collator,
            'compute_metrics': compute_metrics_macro_f1
        }

        for tokenizer_key in ['tokenizer', 'processing_class', None]:
            try:
                candidate_kwargs = dict(trainer_base_kwargs)
                if tokenizer_key is not None:
                    candidate_kwargs[tokenizer_key] = tokenizer
                trainer = Trainer(**candidate_kwargs)
                break
            except TypeError as e:
                trainer_error = e

        if trainer is None:
            raise trainer_error

        for cb in list(trainer.callback_handler.callbacks):
            if cb.__class__.__name__ == 'NotebookProgressCallback':
                trainer.remove_callback(cb.__class__)

        return trainer

    # Try requested mode first; fallback to non-4bit LoRA only when backend is unstable.
    model = build_peft_model(use_4bit=qlora_4bit)
    training_args = build_training_args(prefer_paged_optim=qlora_4bit)
    trainer = build_trainer(model, training_args)

    try:
        trainer.train()
    except AssertionError:
        print('4-bit backend assertion hit. Falling back to non-4bit LoRA for this fold.')
        torch.cuda.empty_cache()
        model = build_peft_model(use_4bit=False)
        training_args = build_training_args(prefer_paged_optim=False)
        trainer = build_trainer(model, training_args)
        trainer.train()

    dev_metrics = trainer.evaluate(dev_ds)
    test_metrics = trainer.evaluate(test_ds, metric_key_prefix='test')

    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'Saved QLoRA adapter and tokenizer to: {output_dir}')

    return trainer, dev_metrics, test_metrics


def run_bert_base_qlora_kfold(train_sentence, train_aspect, train_sentiment,
                              dev_sentence, dev_aspect, dev_sentiment,
                              test_sentence, test_aspect, test_sentiment,
                              numclasses=3,
                              model_name='bert-base-uncased',
                              max_length=128,
                              k_folds=5,
                              output_prefix='kfold_bert_base_qlora',
                              # Pass-through tuning knobs
                              num_train_epochs=6,
                              learning_rate=5e-5,
                              per_device_train_batch_size=8,
                              per_device_eval_batch_size=16,
                              gradient_accumulation_steps=2,
                              warmup_ratio=0.06,
                              weight_decay=0.01,
                              qlora_4bit=True,
                              lora_r=16,
                              lora_alpha=32,
                              lora_dropout=0.05):
    """Run K-fold CV for QLoRA using train+dev data and save each fold adapter."""
    all_sentences = list(train_sentence) + list(dev_sentence)
    all_aspects = list(train_aspect) + list(dev_aspect)
    all_labels = list(train_sentiment) + list(dev_sentiment)

    labels_np = np.asarray(all_labels)
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)

    fold_results = []
    saved_dirs = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_np)), labels_np), start=1):
        print(f"\n{'=' * 72}")
        print(f"TRAINING QLORA BERT-BASE - FOLD {fold_idx}/{k_folds}")
        print(f"{'=' * 72}")

        fold_train_sentence = [all_sentences[i] for i in train_idx]
        fold_train_aspect = [all_aspects[i] for i in train_idx]
        fold_train_sentiment = [all_labels[i] for i in train_idx]

        fold_val_sentence = [all_sentences[i] for i in val_idx]
        fold_val_aspect = [all_aspects[i] for i in val_idx]
        fold_val_sentiment = [all_labels[i] for i in val_idx]

        fold_dir = f"{output_prefix}_fold{fold_idx}"
        _, dev_metrics, test_metrics = run_bert_base_qlora(
            train_sentence=fold_train_sentence,
            train_aspect=fold_train_aspect,
            train_sentiment=fold_train_sentiment,
            dev_sentence=fold_val_sentence,
            dev_aspect=fold_val_aspect,
            dev_sentiment=fold_val_sentiment,
            test_sentence=test_sentence,
            test_aspect=test_aspect,
            test_sentiment=test_sentiment,
            numclasses=numclasses,
            model_name=model_name,
            max_length=max_length,
            output_dir=fold_dir,
            num_train_epochs=num_train_epochs,
            learning_rate=learning_rate,
            per_device_train_batch_size=per_device_train_batch_size,
            per_device_eval_batch_size=per_device_eval_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            warmup_ratio=warmup_ratio,
            weight_decay=weight_decay,
            qlora_4bit=qlora_4bit,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout
        )

        fold_results.append({
            'fold': fold_idx,
            'dev_accuracy': dev_metrics.get('eval_accuracy', None),
            'dev_macro_f1': dev_metrics.get('eval_macro_f1', None),
            'test_accuracy': test_metrics.get('test_accuracy', None),
            'test_macro_f1': test_metrics.get('test_macro_f1', None)
        })
        saved_dirs.append(fold_dir)

    print("\n" + "=" * 72)
    print("QLORA KFOLD SUMMARY")
    print("=" * 72)
    for row in fold_results:
        print(
            f"Fold {row['fold']} | Dev Acc: {row['dev_accuracy']} | Dev F1: {row['dev_macro_f1']} "
            f"| Test Acc: {row['test_accuracy']} | Test F1: {row['test_macro_f1']}"
        )

    valid_test_acc = [x['test_accuracy'] for x in fold_results if x['test_accuracy'] is not None]
    valid_test_f1 = [x['test_macro_f1'] for x in fold_results if x['test_macro_f1'] is not None]
    if len(valid_test_acc) > 0:
        print(f"Average Test Accuracy over {k_folds} folds: {np.mean(valid_test_acc):.4f}")
    if len(valid_test_f1) > 0:
        print(f"Average Test Macro-F1 over {k_folds} folds: {np.mean(valid_test_f1):.4f}")

    return fold_results, saved_dirs

In [ ]:
# Run tuned QLoRA 5-fold CV (BERT-base)
if qlora_ready and torch.cuda.is_available():
    qlora_kfold_results, qlora_saved_dirs = run_bert_base_qlora_kfold(
        train_sentence=train_sentence,
        train_aspect=train_aspect,
        train_sentiment=train_sentiment,
        dev_sentence=dev_sentence,
        dev_aspect=dev_aspect,
        dev_sentiment=dev_sentiment,
        test_sentence=test_sentence,
        test_aspect=test_aspect,
        test_sentiment=test_sentiment,
        numclasses=numclasses,
        model_name='bert-base-uncased',
        max_length=maxlen,
        k_folds=5,
        output_prefix='kfold_bert_base_qlora',
        # Tuned settings for better quality on ABSA
        num_train_epochs=6,
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        weight_decay=0.01,
        qlora_4bit=True,
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05
    )
else:
    print('Skipping QLoRA KFold run: CUDA GPU and required packages are needed.')


TRAINING QLORA BERT-BASE - FOLD 1/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1310.42it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

trainable params: 887,043 || all params: 110,371,590 || trainable%: 0.8037
Using TrainingArguments with eval_strategy and optim=paged_adamw_8bit


d:\System\Conda\envs\absa\lib\site-packages\bitsandbytes\nn\modules.py:409: UserWarning: FP4 quantization state not initialized. Please call .cuda() or .to(device) on the LinearFP4 layer first.
  warnings.warn(


4-bit backend assertion hit. Falling back to non-4bit LoRA for this fold.


...

trainable params: 887,043 || all params: 110,371,590 || trainable%: 0.8037
Using TrainingArguments with eval_strategy and optim=paged_adamw_8bit


d:\System\Conda\envs\absa\lib\site-packages\bitsandbytes\nn\modules.py:409: UserWarning: FP4 quantization state not initialized. Please call .cuda() or .to(device) on the LinearFP4 layer first.
  warnings.warn(


4-bit backend assertion hit. Falling back to non-4bit LoRA for this fold.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8875.37it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

trainable params: 887,043 || all params: 110,371,590 || trainable%: 0.8037
Using TrainingArguments with eval_strategy and optim=adamw_torch
{'loss': '2.401', 'grad_norm': '5.847', 'learning_rate': '2.262e-05', 'epoch': '0.1724'}
{'loss': '2.244', 'grad_norm': '6.753', 'learning_rate': '4.643e-05', 'epoch': '0.3448'}
{'loss': '2.198', 'grad_norm': '2.951', 'learning_rate': '4.992e-05', 'epoch': '0.5172'}
{'loss': '2.16', 'grad_norm': '7.184', 'learning_rate': '4.961e-05', 'epoch': '0.6897'}
{'loss': '2.079', 'grad_norm': '3.309', 'learning_rate': '4.907e-05', 'epoch': '0.8621'}
{'eval_loss': '1.045', 'eval_accuracy': '0.4394', 'eval_macro_f1': '0.3228', 'eval_runtime': '0.5532', 'eval_samples_per_second': '835.1', 'eval_steps_per_second': '52.42', 'epoch': '1'}
{'loss': '2.146', 'grad_norm': '3.38', 'learning_rate': '4.831e-05', 'epoch': '1.034'}
{'loss': '2.011', 'grad_norm': '5.969', 'learning_rate': '4.733e-05', 'epoch': '1.207'}
{'loss': '2.028', 'grad_norm': '5.518', 'learning_rate

In [70]:
# Final comparison table: all 5 models (4 neural + QLoRA)
def _to_percent(x):
    if x is None:
        return None
    x = float(x)
    return round(x * 100, 2) if x <= 1.0 else round(x, 2)


comparison_rows = []

# Existing 4 model results from the KFold neural training block
for model_name in ['lstm', 'attention', 'base', 'lstm_attention']:
    if model_name in model_results:
        r = model_results[model_name]
        comparison_rows.append({
            'model': model_name.upper(),
            'train_acc': _to_percent(r.get('train_acc')),
            'val_acc': _to_percent(r.get('val_acc')),
            'test_acc': _to_percent(r.get('test_acc')),
            'kfold_test_acc': _to_percent(r.get('kfold_avg_test_acc')),
            'kfold_test_f1': _to_percent(r.get('kfold_avg_test_f1'))
        })

# QLoRA aggregate row from fold results
if 'qlora_kfold_results' in globals() and qlora_kfold_results:
    q_test_acc = [x.get('test_accuracy') for x in qlora_kfold_results if x.get('test_accuracy') is not None]
    q_test_f1 = [x.get('test_macro_f1') for x in qlora_kfold_results if x.get('test_macro_f1') is not None]
    q_dev_acc = [x.get('dev_accuracy') for x in qlora_kfold_results if x.get('dev_accuracy') is not None]

    comparison_rows.append({
        'model': 'QLORA_BERT_BASE',
        'train_acc': None,
        'val_acc': _to_percent(np.mean(q_dev_acc)) if len(q_dev_acc) > 0 else None,
        'test_acc': _to_percent(np.mean(q_test_acc)) if len(q_test_acc) > 0 else None,
        'kfold_test_acc': _to_percent(np.mean(q_test_acc)) if len(q_test_acc) > 0 else None,
        'kfold_test_f1': _to_percent(np.mean(q_test_f1)) if len(q_test_f1) > 0 else None
    })

print('\n' + '=' * 108)
print('FINAL MODEL COMPARISON (ALL 5 MODELS)')
print('=' * 108)
print(f"{'Model':<22} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12} {'KFold Test Acc':<16} {'KFold Test F1':<16}")
print('-' * 108)

for row in comparison_rows:
    print(
        f"{row['model']:<22} "
        f"{str(row['train_acc']):<12} "
        f"{str(row['val_acc']):<12} "
        f"{str(row['test_acc']):<12} "
        f"{str(row['kfold_test_acc']):<16} "
        f"{str(row['kfold_test_f1']):<16}"
    )

if len(comparison_rows) == 0:
    print('No metrics found. Run training/evaluation cells first.')


FINAL MODEL COMPARISON (ALL 5 MODELS)
Model                  Train Acc    Val Acc      Test Acc     KFold Test Acc   KFold Test F1   
------------------------------------------------------------------------------------------------------------
LSTM                   88.27        74.14        77.74        75.8             69.0            
ATTENTION              87.79        75.86        77.43        76.65            70.8            
BASE                   89.76        71.98        76.96        76.24            71.2            
LSTM_ATTENTION         87.41        75.43        78.21        76.3             70.4            
QLORA_BERT_BASE        None         69.0         65.8         65.8             48.05           
